# Baseline 1 — BVP + SVM (Widar3.0 TPAMI)

**Goal:** Reproduce the Widar3.0 TPAMI SVM baseline (89.7%) and measure how much
it drops when only 25% of labels are available.

**Protocol:** 3-fold Leave-One-Room-Out (LORO), standard-6 gestures only.

**Feature:** Flattened BVP cube (20×20×20 → 8 000-dim vector), z-score normalised
(already done in preprocess.py), then StandardScaler per fold.

**Two runs per fold:**
| Run | Train set | Purpose |
|---|---|---|
| A | 25% labeled | Our semi-supervised setting |
| B | 100% labeled (25% + 75%) | Reproduce Widar3.0 89.7% |

**Expected runtime:** ~5 min per fold on Colab CPU (RBF SVM, ~5 k–22 k samples).

## 1 — Colab Setup

In [ ]:
# Mount Google Drive (preprocessed.npz lives here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/pi-ssl'

if os.path.exists(REPO_DIR):
    # Pull latest changes if repo already cloned
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Fatima112299/pi-ssl.git {REPO_DIR}

# Make project importable
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2 — Configuration

**Only change `NPZ_PATH`** to wherever you uploaded `preprocessed.npz` on your Drive.

In [ ]:
# ── Path to your preprocessed.npz on Google Drive ────────────────────────────
NPZ_PATH = '/content/drive/MyDrive/pi-ssl/preprocessed.npz'

# ── SVM hyperparameters ───────────────────────────────────────────────────────
# RBF SVM matches the Widar3.0 TPAMI paper.
# If training is too slow, switch kernel to 'linear'.
SVM_KERNEL = 'rbf'
SVM_C      = 10.0
SVM_GAMMA  = 'scale'   # 1 / (n_features * X.var())

RANDOM_SEED    = 42
LABELED_RATIO  = 0.25
# ─────────────────────────────────────────────────────────────────────────────

print(f'NPZ_PATH : {NPZ_PATH}')
print(f'SVM      : kernel={SVM_KERNEL}  C={SVM_C}  gamma={SVM_GAMMA}')

## 3 — Imports

In [ ]:
import sys, os

# Ensure repo is on the path regardless of cell execution order
REPO_DIR = '/content/pi-ssl'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if not os.path.exists(REPO_DIR):
    raise RuntimeError('Repo not found — run the clone cell first, then re-run this cell.')

import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from src.data.bvp_dataset import load_npz
from src.data.splits import make_loeo_splits
from src.data.widar3_dataset import ID_TO_GESTURE, STANDARD_6_GESTURES

print('All imports OK.')

## 4 — Load Data

In [ ]:
print('Loading preprocessed.npz ...')
t0 = time.time()
npz_data, file_list = load_npz(NPZ_PATH)
print(f'  Samples   : {len(file_list)}')
print(f'  BVP shape : {npz_data["bvp"].shape}  {npz_data["bvp"].dtype}')
print(f'  Load time : {time.time() - t0:.1f}s')

## 5 — Helper Functions

In [ ]:
def extract_features(file_subset, bvp_array):
    """
    Flatten each (20, 20, 20) BVP cube into an 8 000-dim feature vector.

    WHY flatten: SVMs operate on fixed-length vectors. The BVP cube is
    already normalised and temporally aligned, so the raw flattened values
    carry the full spectral + spatial + temporal information without any
    hand-crafted feature engineering — matching the Widar3.0 TPAMI approach.

    Returns
    -------
    X : float32 ndarray (N, 8000)
    y : int32   ndarray (N,)  — global gesture IDs (0-5 for standard-6)
    """
    indices = np.array([f['npz_idx'] for f in file_subset])
    X = bvp_array[indices]              # (N, 20, 20, 20)
    X = X.reshape(len(X), -1)           # (N, 8000)
    y = np.array([f['gesture_id'] for f in file_subset], dtype=np.int32)
    return X, y


def run_svm_fold(fold, npz_data, file_list,
                 labeled_ratio=0.25, seed=42,
                 kernel='rbf', C=10.0, gamma='scale'):
    """
    Train and evaluate SVM for one LORO fold.

    Returns a dict with accuracy for both 25% and 100% labeled settings.
    """
    print(f'\n{"="*60}')
    print(f'FOLD {fold}')
    print(f'{"="*60}')

    # ── Build splits ──────────────────────────────────────────────────────────
    _, labeled, unlabeled, test = make_loeo_splits(
        bvp_root=None, fold=fold,
        labeled_ratio=labeled_ratio, seed=seed,
        file_list=file_list
    )
    full_train = labeled + unlabeled   # 100% labeled training set

    print(f'  Labeled train (25%) : {len(labeled)}')
    print(f'  Full train   (100%) : {len(full_train)}')
    print(f'  Test                : {len(test)}')

    # ── Feature extraction ────────────────────────────────────────────────────
    X_lab,  y_lab  = extract_features(labeled,    npz_data['bvp'])
    X_full, y_full = extract_features(full_train, npz_data['bvp'])
    X_test, y_test = extract_features(test,       npz_data['bvp'])

    results = {}

    for tag, X_tr, y_tr in [
        ('25%_labeled', X_lab,  y_lab),
        ('100%_labeled', X_full, y_full),
    ]:
        print(f'\n  [{tag}]  train={len(y_tr)}  test={len(y_test)}')

        # StandardScaler: fit on train only, apply to test
        # WHY: prevents test statistics from leaking into training
        scaler = StandardScaler()
        X_tr_s   = scaler.fit_transform(X_tr)
        X_test_s = scaler.transform(X_test)

        # Train SVM
        svm = SVC(kernel=kernel, C=C, gamma=gamma,
                  decision_function_shape='ovr',
                  cache_size=2000, random_state=seed)
        t_train = time.time()
        svm.fit(X_tr_s, y_tr)
        train_time = time.time() - t_train
        print(f'    Train time : {train_time:.1f}s')

        # Evaluate
        y_pred = svm.predict(X_test_s)
        acc    = accuracy_score(y_test, y_pred) * 100
        print(f'    Accuracy   : {acc:.2f}%')

        # Per-class breakdown
        labels_present = sorted(set(y_test.tolist()))
        class_names    = [ID_TO_GESTURE[i] for i in labels_present]
        print(f'    Per-class breakdown:')
        report = classification_report(
            y_test, y_pred,
            labels=labels_present, target_names=class_names,
            digits=3, zero_division=0
        )
        for line in report.strip().split('\n'):
            print(f'      {line}')

        results[tag] = {
            'accuracy'   : acc,
            'train_time' : train_time,
            'y_pred'     : y_pred,
            'y_test'     : y_test,
        }

    return results


print('Helper functions defined.')

## 6 — Run 3-Fold Evaluation

Each fold trains two SVMs: one with 25% labels (our setting) and one
with 100% labels (to reproduce the Widar3.0 89.7% baseline).

> **Time estimate:** ~5 min/fold on Colab CPU. All 3 folds ≈ 15–20 min.

In [ ]:
fold_results = {}
t_total = time.time()

for fold in range(3):
    fold_results[fold] = run_svm_fold(
        fold       = fold,
        npz_data   = npz_data,
        file_list  = file_list,
        labeled_ratio = LABELED_RATIO,
        seed       = RANDOM_SEED,
        kernel     = SVM_KERNEL,
        C          = SVM_C,
        gamma      = SVM_GAMMA,
    )

print(f'\nTotal wall time: {(time.time()-t_total)/60:.1f} min')

## 7 — Summary Table

In [ ]:
FOLD_ROOMS = {0: 'Office', 1: 'Classroom', 2: 'Hall'}

accs_25  = [fold_results[f]['25%_labeled']['accuracy']  for f in range(3)]
accs_100 = [fold_results[f]['100%_labeled']['accuracy'] for f in range(3)]

mean_25  = np.mean(accs_25)
std_25   = np.std(accs_25)
mean_100 = np.mean(accs_100)
std_100  = np.std(accs_100)

print('\n' + '='*65)
print(f'{"RESULTS SUMMARY":^65}')
print('='*65)
print(f'{"Fold":<6} {"Test Room":<12} {"SVM 25% labels":>16} {"SVM 100% labels":>17}')
print('-'*65)
for f in range(3):
    a25  = fold_results[f]['25%_labeled']['accuracy']
    a100 = fold_results[f]['100%_labeled']['accuracy']
    print(f'{f:<6} {FOLD_ROOMS[f]:<12} {a25:>15.2f}%  {a100:>15.2f}%')
print('-'*65)
print(f'{"Mean":<18} {mean_25:>15.2f}%  {mean_100:>15.2f}%')
print(f'{"Std":<18} {std_25:>15.2f}%  {std_100:>15.2f}%')
print('='*65)
print()
print('Reference baselines:')
print(f'  Widar3.0 TPAMI (SVM, 100% labels, same protocol) : 89.70%')
print(f'  SenseFi CNN-5  (random split, 22 classes)         : 70.19%')
print()
print(f'Gap to close with PI-SSL (25% labels vs Widar3.0 100% labels):')
print(f'  {mean_100 - mean_25:+.2f}%  ({mean_25:.2f}% → {mean_100:.2f}%)')

## 8 — Per-Gesture Accuracy Breakdown

In [ ]:
# Show per-gesture accuracy averaged across folds for the 25% labeled setting
from collections import defaultdict

gesture_correct = defaultdict(int)
gesture_total   = defaultdict(int)

for f in range(3):
    y_test = fold_results[f]['25%_labeled']['y_test']
    y_pred = fold_results[f]['25%_labeled']['y_pred']
    for true, pred in zip(y_test, y_pred):
        gesture_total[true]   += 1
        gesture_correct[true] += int(true == pred)

print('Per-gesture accuracy (25% labeled, averaged across 3 folds):')
print(f'  {"Gesture":>22}   {"Correct":>8}   {"Total":>7}   {"Accuracy":>9}')
print(f'  {"-"*56}')
for gid in sorted(gesture_total):
    name = ID_TO_GESTURE[gid]
    acc  = 100 * gesture_correct[gid] / gesture_total[gid]
    print(f'  {name:>22}   {gesture_correct[gid]:>8}   '
          f'{gesture_total[gid]:>7}   {acc:>8.2f}%')

## 9 — Save Results

In [ ]:
import json

# Save a clean summary to Drive for later use in the paper
RESULTS_PATH = '/content/drive/MyDrive/pi-ssl/results_svm_baseline.json'

summary = {
    'method'         : 'BVP+SVM',
    'kernel'         : SVM_KERNEL,
    'C'              : SVM_C,
    'gamma'          : SVM_GAMMA,
    'labeled_ratio'  : LABELED_RATIO,
    'seed'           : RANDOM_SEED,
    'folds': {
        str(f): {
            'test_room'       : FOLD_ROOMS[f],
            'acc_25pct'       : fold_results[f]['25%_labeled']['accuracy'],
            'acc_100pct'      : fold_results[f]['100%_labeled']['accuracy'],
            'train_time_25pct': fold_results[f]['25%_labeled']['train_time'],
            'train_time_100pct': fold_results[f]['100%_labeled']['train_time'],
        }
        for f in range(3)
    },
    'mean_acc_25pct'  : float(mean_25),
    'std_acc_25pct'   : float(std_25),
    'mean_acc_100pct' : float(mean_100),
    'std_acc_100pct'  : float(std_100),
}

with open(RESULTS_PATH, 'w') as fp:
    json.dump(summary, fp, indent=2)

print(f'Results saved to {RESULTS_PATH}')
print(json.dumps(summary, indent=2))

---
## Interpretation

| | Mean Accuracy |
|---|---|
| **SVM — 25% labels** (our semi-supervised setting) | `mean_25` % |
| **SVM — 100% labels** (Widar3.0 TPAMI reproduction) | `mean_100` % |
| **Widar3.0 TPAMI reported** | 89.70% |

**What this tells us:**
- If `mean_100 ≈ 89.7%` → our preprocessing and protocol are correct, the reproduction is valid.
- The gap `mean_100 − mean_25` is how much accuracy is lost from label reduction.
- PI-SSL's goal: close that gap — reach `>89.5%` using only the 25% labeled set.

**Next notebook:** `baseline_cnn5.ipynb` — supervised CNN-5 baseline on the same splits.